# Module 1: Predictive Modeling for D&O Insurance
## Using Generalized Linear Models and XGBoost

**Course:** Predictive Analytics for Insurance  
**Module:** 1 of 6  
**Duration:** 90 minutes  
**Prerequisites:** Basic Python, pandas, understanding of regression

---

## Learning Objectives

By the end of this module, you will be able to:
1. Load and explore D&O insurance claims data
2. Perform exploratory data analysis (EDA) on insurance datasets
3. Clean and prepare data for predictive modeling
4. Split data into training, validation, and test sets
5. Build and interpret a logistic regression model (GLM)
6. Build and compare an XGBoost model
7. Evaluate model performance using insurance-relevant metrics

## 1. Introduction to D&O Insurance

**Directors and Officers (D&O) Insurance** protects company leadership from personal losses if they are sued for decisions made while running the company.

### Key Concepts:
- **Loss Occurrence**: Whether a claim was filed (binary: Yes/No)
- **Severity**: How much the claim cost (continuous: $0+)
- **Risk Factors**: Company size, industry, revenue, prior claims, etc.

### Our Goal:
Build a **binary classification model** to predict whether a D&O policy will result in a claim.

### Why GLM?
- **Interpretable**: Coefficients show feature importance
- **Industry standard**: Widely used in insurance
- **Regulatory friendly**: Transparent for audits

### Why Also XGBoost?
- **Better performance**: Often more accurate
- **Handles complex interactions**: Captures non-linear relationships
- **Comparison baseline**: See if complexity is worth it

## 2. Data Loading & Initial Exploration

### 2.1 Setup Environment

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix, 
    classification_report, roc_curve
)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries imported successfully")

### 2.2 Generate Synthetic D&O Insurance Data

For this lesson, we'll create realistic synthetic data. In practice, you'd load from CSV/database.

In [ ]:
def generate_do_insurance_data(n_samples=2000, random_state=42):
    """
    Generate synthetic D&O insurance data
    
    Features:
    - company_age: Years in business (1-100)
    - revenue_millions: Annual revenue in millions
    - num_directors: Number of board directors (3-15)
    - prior_claims: Number of prior claims (0-5)
    - industry: Industry sector (Tech, Finance, Healthcare, Manufacturing, Retail)
    - employees: Number of employees (10-10000)
    - public_company: Public vs Private (0/1)
    
    Target:
    - claim_occurred: Whether a claim was filed (0/1)
    """
    np.random.seed(random_state)
    
    # Generate features
    data = {
        'company_age': np.random.randint(1, 101, n_samples),
        'revenue_millions': np.random.exponential(50, n_samples),
        'num_directors': np.random.randint(3, 16, n_samples),
        'prior_claims': np.random.poisson(0.5, n_samples),
        'employees': np.random.lognormal(5, 2, n_samples).astype(int),
        'public_company': np.random.binomial(1, 0.3, n_samples),
        'industry': np.random.choice(
            ['Tech', 'Finance', 'Healthcare', 'Manufacturing', 'Retail'], 
            n_samples, 
            p=[0.25, 0.20, 0.20, 0.20, 0.15]
        )
    }
    
    df = pd.DataFrame(data)
    
    # Clip values to realistic ranges
    df['revenue_millions'] = df['revenue_millions'].clip(0.5, 500)
    df['employees'] = df['employees'].clip(10, 10000)
    df['prior_claims'] = df['prior_claims'].clip(0, 5)
    
    # Generate target variable (claim occurrence) based on risk factors
    risk_score = (
        0.005 * df['revenue_millions'] +          # Bigger companies = more risk
        0.02 * df['num_directors'] +              # More directors = more risk
        0.15 * df['prior_claims'] +               # Prior claims = strong predictor
        0.08 * df['public_company'] +             # Public companies = more risk
        -0.002 * df['company_age'] +              # Older companies = slightly less risk
        0.00001 * df['employees']                 # Employee count impact
    )
    
    # Industry risk adjustments
    industry_risk = {
        'Finance': 0.3,      # High risk
        'Tech': 0.2,         # Medium-high risk
        'Healthcare': 0.15,  # Medium risk
        'Manufacturing': 0.1,# Medium-low risk
        'Retail': 0.05       # Lower risk
    }
    risk_score += df['industry'].map(industry_risk)
    
    # Convert to probability using sigmoid
    probability = 1 / (1 + np.exp(-risk_score))
    probability = probability.clip(0.05, 0.95)
    
    # Generate binary outcome
    df['claim_occurred'] = np.random.binomial(1, probability)
    
    return df

# Generate data
df = generate_do_insurance_data(n_samples=2000, random_state=42)

print("✅ Data generated successfully")
print(f"Dataset shape: {df.shape}")

### 2.3 First Look at the Data

In [ ]:
# Display first few rows
print("First 5 rows:")
df.head()

In [ ]:
# Dataset info
print("Dataset Info:")
df.info()

In [ ]:
# Basic statistics
print("Basic Statistics:")
df.describe()

In [ ]:
# Target variable distribution
print("Target Variable Distribution:")
print(df['claim_occurred'].value_counts())
print(f"\nClaim Rate: {df['claim_occurred'].mean():.1%}")

## 3. Exploratory Data Analysis

### 3.1 Check for Missing Values

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print("\nMissing Value Percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

### 3.2 Target Variable Distribution

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df['claim_occurred'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('Claim Occurrence Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Claim Occurred')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Claim', 'Claim'], rotation=0)

# Pie chart
df['claim_occurred'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                          colors=['skyblue', 'salmon'])
axes[1].set_title('Claim Rate', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"Total Records: {len(df)}")
print(f"Claims: {df['claim_occurred'].sum()}")
print(f"No Claims: {(df['claim_occurred'] == 0).sum()}")
print(f"Claim Rate: {df['claim_occurred'].mean():.1%}")

### 3.3 Numerical Features Distribution

In [ ]:
# Plot distributions of numerical features
numerical_cols = ['company_age', 'revenue_millions', 'num_directors', 
                  'prior_claims', 'employees']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    axes[idx].hist(df[col], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{col.replace("_", " ").title()}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')

# Remove extra subplot
axes[-1].axis('off')

plt.tight_layout()
plt.show()

### 3.4 Categorical Features Distribution

In [ ]:
# Industry distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Industry counts
industry_counts = df['industry'].value_counts()
axes[0].bar(industry_counts.index, industry_counts.values, color='teal', alpha=0.7)
axes[0].set_title('Companies by Industry', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Industry')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Public vs Private
public_counts = df['public_company'].value_counts()
axes[1].bar(['Private', 'Public'], public_counts.values, color=['coral', 'skyblue'], alpha=0.7)
axes[1].set_title('Company Type Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Company Type')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### 3.5 Feature Relationships with Target

In [ ]:
# Claim rate by industry
claim_by_industry = df.groupby('industry')['claim_occurred'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
claim_by_industry.plot(kind='bar', color='crimson', alpha=0.7)
plt.title('Claim Rate by Industry', fontsize=14, fontweight='bold')
plt.xlabel('Industry')
plt.ylabel('Claim Rate')
plt.xticks(rotation=45)
plt.axhline(df['claim_occurred'].mean(), color='black', linestyle='--', 
            label=f'Overall Rate: {df["claim_occurred"].mean():.1%}')
plt.legend()
plt.tight_layout()
plt.show()

print("Claim Rate by Industry:")
print(claim_by_industry.apply(lambda x: f"{x:.1%}"))

In [ ]:
# Box plots: Numerical features by claim status
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    df.boxplot(column=col, by='claim_occurred', ax=axes[idx])
    axes[idx].set_title(f'{col.replace("_", " ").title()}')
    axes[idx].set_xlabel('Claim Occurred (0=No, 1=Yes)')
    axes[idx].set_ylabel(col)
    
axes[-1].axis('off')
plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

### 3.6 Correlation Analysis

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))

# Create correlation matrix (numerical features only)
corr_data = df[numerical_cols + ['claim_occurred']].corr()

sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with Target (claim_occurred):")
print(corr_data['claim_occurred'].sort_values(ascending=False))

## 4. Data Cleaning & Preprocessing

### 4.1 Handle Outliers

In [ ]:
def detect_outliers_iqr(df, columns):
    """Detect outliers using IQR method"""
    outliers = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outliers[col] = outlier_mask.sum()
        
    return pd.Series(outliers)

# Check for outliers
outlier_counts = detect_outliers_iqr(df, numerical_cols)
print("Outlier Counts by Feature:")
print(outlier_counts)

# For this exercise, we'll keep outliers (they may be legitimate high-risk cases)
# In practice, investigate each case

### 4.2 Create Derived Features

In [ ]:
# Feature engineering
df['revenue_per_employee'] = df['revenue_millions'] * 1_000_000 / df['employees']
df['directors_per_100_employees'] = (df['num_directors'] / df['employees']) * 100
df['has_prior_claims'] = (df['prior_claims'] > 0).astype(int)

print("New features created:")
df[['revenue_per_employee', 'directors_per_100_employees', 'has_prior_claims']].head()

### 4.3 Encode Categorical Variables

In [ ]:
# One-hot encode industry
df_encoded = pd.get_dummies(df, columns=['industry'], prefix='industry', drop_first=True)

print(f"Original shape: {df.shape}")
print(f"After encoding: {df_encoded.shape}")
print(f"\nNew columns: {df_encoded.columns.tolist()}")

### 4.4 Separate Features and Target

In [ ]:
# Separate features and target
X = df_encoded.drop('claim_occurred', axis=1)
y = df_encoded['claim_occurred']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")

## 5. Train/Validation/Test Split

### 5.1 Understanding the Split Strategy

**Why 3-way split?**

1. **Training Set (60%)**: Learn patterns
2. **Validation Set (20%)**: Tune hyperparameters, select features
3. **Test Set (20%)**: Final evaluation (DO NOT touch until end!)

**Insurance Context:**
- Need sufficient data in each set for stable estimates
- Ensure claim rate is similar across splits (stratification)

### 5.2 Perform the Split

In [ ]:
# First split: Train vs (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.4,  # 40% for validation + test
    random_state=42,
    stratify=y  # Maintain claim rate in each split
)

# Second split: Validation vs Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,  # Split the 40% in half
    random_state=42,
    stratify=y_temp
)

print("Data Split Summary:")
print(f"Training set:   {len(X_train):4d} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation set: {len(X_val):4d} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test set:       {len(X_test):4d} samples ({len(X_test)/len(X)*100:.1f}%)")

print("\nClaim Rate by Set:")
print(f"Training:   {y_train.mean():.1%}")
print(f"Validation: {y_val.mean():.1%}")
print(f"Test:       {y_test.mean():.1%}")

### 5.3 Scale Features

In [ ]:
# Initialize scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames for interpretability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print("✅ Features scaled successfully")
print("\nScaled feature statistics (training set):")
print(X_train_scaled.describe().loc[['mean', 'std']])

## 6. Logistic Regression (GLM)

### 6.1 Understanding Logistic Regression

**Logistic Regression for Insurance:**

Formula: `P(claim) = 1 / (1 + e^-(β0 + β1*x1 + β2*x2 + ... + βn*xn))`

**Key Properties:**
- Linear combination of features
- Output between 0 and 1 (probability)
- Interpretable coefficients
- Assumes linear relationship between log-odds and features

**Insurance Use Cases:**
- Claim frequency modeling
- Risk classification
- Premium pricing
- Regulatory compliance (transparent model)

### 6.2 Train Logistic Regression Model

In [ ]:
# Initialize model
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='lbfgs',  # Good for small-medium datasets
    penalty='l2',     # L2 regularization
    C=1.0             # Regularization strength
)

# Train the model
log_reg.fit(X_train_scaled, y_train)

print("✅ Logistic Regression model trained")
print(f"Number of iterations: {log_reg.n_iter_[0]}")
print(f"Converged: {log_reg.n_iter_[0] < log_reg.max_iter}")

### 6.3 Model Coefficients (Feature Importance)

In [ ]:
# Extract coefficients
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg.coef_[0],
    'Abs_Coefficient': np.abs(log_reg.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("Top 10 Most Important Features:")
print(coefficients.head(10))

In [ ]:
# Visualize coefficients
plt.figure(figsize=(12, 8))
top_features = coefficients.head(15)
colors = ['green' if x > 0 else 'red' for x in top_features['Coefficient']]
plt.barh(range(len(top_features)), top_features['Coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Coefficient Value')
plt.title('Top 15 Feature Coefficients (Logistic Regression)', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Positive coefficient = increases claim probability")
print("- Negative coefficient = decreases claim probability")
print("- Larger magnitude = stronger effect")

### 6.4 Make Predictions

In [ ]:
# Predict on validation set
y_val_pred_proba = log_reg.predict_proba(X_val_scaled)[:, 1]  # Probabilities
y_val_pred = log_reg.predict(X_val_scaled)  # Binary predictions

# Show first 10 predictions
results_df = pd.DataFrame({
    'Actual': y_val.values[:10],
    'Predicted_Prob': y_val_pred_proba[:10],
    'Predicted_Class': y_val_pred[:10]
})

print("First 10 Predictions:")
results_df

### 6.5 Evaluate on Validation Set

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
auc = roc_auc_score(y_val, y_val_pred_proba)

print("Logistic Regression - Validation Set Performance:")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f} (Of predicted claims, {precision:.1%} were correct)")
print(f"Recall:    {recall:.3f} (Of actual claims, {recall:.1%} were caught)")
print(f"F1-Score:  {f1:.3f}")
print(f"AUC-ROC:   {auc:.3f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val, y_val_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Claim', 'Claim'],
            yticklabels=['No Claim', 'Claim'])
plt.title('Confusion Matrix - Logistic Regression', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print("Confusion Matrix Breakdown:")
print(f"True Negatives:  {cm[0,0]} (Correctly predicted no claim)")
print(f"False Positives: {cm[0,1]} (Predicted claim, but no claim)")
print(f"False Negatives: {cm[1,0]} (Predicted no claim, but claim occurred)")
print(f"True Positives:  {cm[1,1]} (Correctly predicted claim)")

### 6.6 ROC Curve

In [ ]:
# Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_val, y_val_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. XGBoost Model

### 7.1 Understanding XGBoost

**XGBoost for Insurance:**

**What is it?**
- Gradient Boosting Decision Trees
- Builds trees sequentially, each correcting previous errors
- Highly accurate, handles non-linear relationships

**Pros:**
- Better accuracy than GLM
- Handles interactions automatically
- Less sensitive to scaling
- Built-in feature importance

**Cons:**
- Less interpretable ("black box")
- Prone to overfitting
- More computationally expensive
- Harder to explain to regulators

**When to use in insurance:**
- When accuracy is critical
- As a benchmark to beat
- For ensemble models

### 7.2 Train XGBoost Model

In [ ]:
# Initialize XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,        # Number of trees
    max_depth=5,             # Maximum tree depth
    learning_rate=0.1,       # Step size shrinkage
    subsample=0.8,           # Fraction of samples per tree
    colsample_bytree=0.8,    # Fraction of features per tree
    random_state=42,
    eval_metric='logloss'    # Evaluation metric
)

# Train model (no scaling needed for XGBoost)
xgb_model.fit(X_train, y_train)

print("✅ XGBoost model trained")

### 7.3 Feature Importance (XGBoost)

In [ ]:
# Get feature importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Most Important Features (XGBoost):")
print(importance.head(10))

In [ ]:
# Visualize
plt.figure(figsize=(12, 8))
top_features = importance.head(15)
plt.barh(range(len(top_features)), top_features['Importance'], color='steelblue', alpha=0.7)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Importance Score')
plt.title('Top 15 Feature Importances (XGBoost)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.4 Make Predictions

In [ ]:
# Predict on validation set
y_val_pred_xgb_proba = xgb_model.predict_proba(X_val)[:, 1]
y_val_pred_xgb = xgb_model.predict(X_val)

print("First 10 XGBoost Predictions:")
results_df_xgb = pd.DataFrame({
    'Actual': y_val.values[:10],
    'Predicted_Prob': y_val_pred_xgb_proba[:10],
    'Predicted_Class': y_val_pred_xgb[:10]
})
results_df_xgb

### 7.5 Evaluate on Validation Set

In [ ]:
# Calculate metrics
accuracy_xgb = accuracy_score(y_val, y_val_pred_xgb)
precision_xgb = precision_score(y_val, y_val_pred_xgb)
recall_xgb = recall_score(y_val, y_val_pred_xgb)
f1_xgb = f1_score(y_val, y_val_pred_xgb)
auc_xgb = roc_auc_score(y_val, y_val_pred_xgb_proba)

print("XGBoost - Validation Set Performance:")
print(f"Accuracy:  {accuracy_xgb:.3f}")
print(f"Precision: {precision_xgb:.3f}")
print(f"Recall:    {recall_xgb:.3f}")
print(f"F1-Score:  {f1_xgb:.3f}")
print(f"AUC-ROC:   {auc_xgb:.3f}")

In [ ]:
# Confusion Matrix
cm_xgb = confusion_matrix(y_val, y_val_pred_xgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['No Claim', 'Claim'],
            yticklabels=['No Claim', 'Claim'])
plt.title('Confusion Matrix - XGBoost', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 8. Model Comparison & Evaluation

### 8.1 Compare Metrics Side-by-Side

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'],
    'Logistic Regression': [accuracy, precision, recall, f1, auc],
    'XGBoost': [accuracy_xgb, precision_xgb, recall_xgb, f1_xgb, auc_xgb]
})

comparison['Difference'] = comparison['XGBoost'] - comparison['Logistic Regression']
comparison['Winner'] = comparison['Difference'].apply(
    lambda x: 'XGBoost' if x > 0.01 else ('LogReg' if x < -0.01 else 'Tie')
)

print("Model Comparison (Validation Set):")
comparison

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison['Metric']))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['Logistic Regression'], width, 
               label='Logistic Regression', color='skyblue', alpha=0.8)
bars2 = ax.bar(x + width/2, comparison['XGBoost'], width, 
               label='XGBoost', color='salmon', alpha=0.8)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Metric'])
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 8.2 ROC Curve Comparison

In [ ]:
# Plot both ROC curves
fpr_xgb, tpr_xgb, _ = roc_curve(y_val, y_val_pred_xgb_proba)

plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})', 
         linewidth=2, color='blue')
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {auc_xgb:.3f})', 
         linewidth=2, color='red')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 8.3 Test Set Evaluation (Final Check)

**IMPORTANT:** Only evaluate on test set ONCE at the very end! This prevents "peeking" and gives unbiased performance estimate.

In [ ]:
# Logistic Regression on Test Set
y_test_pred = log_reg.predict(X_test_scaled)
y_test_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

test_accuracy = accuracy_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

print("LOGISTIC REGRESSION - FINAL TEST SET PERFORMANCE:")
print(f"Accuracy: {test_accuracy:.3f}")
print(f"AUC-ROC:  {test_auc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, 
                          target_names=['No Claim', 'Claim']))

In [ ]:
# XGBoost on Test Set
y_test_pred_xgb = xgb_model.predict(X_test)
y_test_pred_xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

test_accuracy_xgb = accuracy_score(y_test, y_test_pred_xgb)
test_auc_xgb = roc_auc_score(y_test, y_test_pred_xgb_proba)

print("="*60)
print("XGBOOST - FINAL TEST SET PERFORMANCE:")
print(f"Accuracy: {test_accuracy_xgb:.3f}")
print(f"AUC-ROC:  {test_auc_xgb:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_xgb, 
                          target_names=['No Claim', 'Claim']))

### 8.4 Business Impact Analysis

In [ ]:
# Assume average claim cost
avg_claim_cost = 100_000  # $100k average claim
premium_revenue = 5_000   # $5k premium per policy

# Calculate costs for each model
def calculate_business_impact(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # False Negative Cost: Missed claims we'll have to pay
    fn_cost = fn * avg_claim_cost
    
    # False Positive Cost: Lost premium revenue from rejected customers
    fp_cost = fp * premium_revenue
    
    # True Positive Benefit: Avoided claims
    tp_benefit = tp * avg_claim_cost
    
    # True Negative Benefit: Premiums from good customers
    tn_benefit = tn * premium_revenue
    
    total_cost = fn_cost + fp_cost
    total_benefit = tp_benefit + tn_benefit
    net_benefit = total_benefit - total_cost
    
    print(f"\n{model_name} - Business Impact (Test Set):")
    print(f"False Negative Cost (Missed Claims): ${fn_cost:,.0f}")
    print(f"False Positive Cost (Lost Revenue):  ${fp_cost:,.0f}")
    print(f"True Positive Benefit (Avoided):     ${tp_benefit:,.0f}")
    print(f"True Negative Benefit (Revenue):     ${tn_benefit:,.0f}")
    print(f"{'='*50}")
    print(f"Net Benefit: ${net_benefit:,.0f}")
    
    return {
        'Model': model_name,
        'FN_Cost': fn_cost,
        'FP_Cost': fp_cost,
        'Net_Benefit': net_benefit
    }

impact_logreg = calculate_business_impact(y_test, y_test_pred, "Logistic Regression")
impact_xgb = calculate_business_impact(y_test, y_test_pred_xgb, "XGBoost")

In [ ]:
# Compare
impact_df = pd.DataFrame([impact_logreg, impact_xgb])
print("="*60)
print("BUSINESS IMPACT COMPARISON:")
impact_df

## 9. Summary & Next Steps

### 9.1 Key Takeaways

**What We Learned:**

1. **Data Exploration**
   - D&O insurance data characteristics
   - Feature distributions and relationships
   - Target variable imbalance

2. **Data Preparation**
   - Train/Validation/Test split (60/20/20)
   - Feature scaling for GLM
   - One-hot encoding for categorical variables

3. **Modeling**
   - **Logistic Regression (GLM):**
     - Interpretable
     - Good baseline
     - Regulatory friendly
   
   - **XGBoost:**
     - Higher accuracy
     - Better at complex patterns
     - Less interpretable

4. **Evaluation**
   - Multiple metrics: Accuracy, Precision, Recall, F1, AUC
   - Business impact: Cost-benefit analysis
   - Test set as final evaluation

**Key Insight:**
XGBoost slightly outperforms Logistic Regression, but the difference may not justify the added complexity depending on business needs.

### 9.2 Practice Exercises

**Try These:**

**1. Easy:**
- Change the train/val/test split to 70/15/15
- Add a new feature: revenue per director
- Plot precision-recall curves

**2. Medium:**
- Tune XGBoost hyperparameters (max_depth, n_estimators)
- Implement different classification thresholds (0.3, 0.5, 0.7)
- Create interaction features (age * revenue)

**3. Hard:**
- Build a stacked ensemble (LogReg + XGBoost)
- Implement SMOTE for class imbalance
- Add time-based features
- Create a pipeline with sklearn Pipeline class

**4. Research:**
- Read about Tweedie GLM for insurance
- Compare with LightGBM and CatBoost
- Investigate SHAP values for XGBoost interpretability

### 9.3 Resources

**Books:**
- "An Introduction to Statistical Learning" - James et al.
- "The Elements of Statistical Learning" - Hastie et al.
- "Predictive Modeling of Insurance Claims" - Frees

**Online Courses:**
- Coursera: Machine Learning Specialization
- DataCamp: Machine Learning with scikit-learn
- Kaggle Learn: Intro to Machine Learning

**Insurance-Specific:**
- SOA (Society of Actuaries) predictive analytics modules
- CAS (Casualty Actuarial Society) resources

**Tools Documentation:**
- scikit-learn: https://scikit-learn.org
- XGBoost: https://xgboost.readthedocs.io
- pandas: https://pandas.pydata.org

---

**End of Module 1**

*Next Module: Hyperparameter Tuning & Advanced Feature Engineering*